In [1]:
from pyspark.mllib.linalg import Vectors

def readVectorsSeq(filename):
    """
    Reads a text file of comma-separated values and returns a list of Spark Vectors.
    """
    vectors_list = []
    
    # Open the file and read
    with open(filename, 'r') as file:
        for line in file:
            # Remove whitespace and split by comma
            string_features = line.strip().split(',')
            
            # Convert the string values to floats
            float_features = [float(x) for x in string_features]
            
            # Create a Spark Dense Vector and add it to our list
            vector = Vectors.dense(float_features)
            vectors_list.append(vector)
            
    return vectors_list


filename = 'Q1- UCI Spam clustering/spambase.data'

try:
    P = readVectorsSeq(filename)
    print(f"Successfully loaded {len(P)} vectors.")
    print(f"Dimension of the first vector: {len(P[0])}")
except FileNotFoundError:
    print(f"Error: Could not find the file '{filename}'. Please check the path.")

Successfully loaded 4601 vectors.
Dimension of the first vector: 58


In [2]:
import time

In [6]:
def kcenter(P, k):
    """
    Computes k centers using the Farthest-First Traversal algorithm.
    Runs in O(|P| * k) time.
    """
    n = len(P)
    if k <= 0:
        return []
    if k >= n:
        return P[:]
    
    C = []
    
    # 1. Pick the first center
    C.append(P[0])
    
    # 2. Keep track of the minimum squared distance from each point to any center in C.
    min_distances = [Vectors.squared_distance(P[i], C[0]) for i in range(n)]
    
    # 3. Find the remaining k-1 centers
    for _ in range(1, k):
        max_dist = -1.0
        farthest_point_idx = -1
        
        # Find the point with the maximum minimum distance to the current centers
        for i in range(n):
            if min_distances[i] > max_dist:
                max_dist = min_distances[i]
                farthest_point_idx = i
                
        # Add the farthest point to our list of centers
        new_center = P[farthest_point_idx]
        C.append(new_center)
        
        # Update min_distances to account for the new center
        for i in range(n):
            dist_to_new = Vectors.squared_distance(P[i], new_center)
            if dist_to_new < min_distances[i]:
                min_distances[i] = dist_to_new
                
    return C


k_test = 5
start_time = time.time()
centers = kcenter(P, k_test)
end_time = time.time()

print(f"Computed {len(centers)} centers in {end_time - start_time:.3f} seconds.")

Computed 5 centers in 0.035 seconds.


In [7]:
import random

In [10]:
def kmeansPP(P, k):
    """
    Computes k centers using the kmeans++ initialization algorithm.
    Runs in O(|P| * k) time.
    """
    n = len(P)
    if k <= 0:
        return []
    if k >= n:
        return P[:]
    
    C = []
    
    # 1. Pick the first center uniformly at random
    first_center_idx = random.randint(0, n - 1)
    C.append(P[first_center_idx])
    
    # 2. Keep track of the minimum squared distance from each point to the closest center.
    min_sq_distances = [P[i].squared_distance(C[0]) for i in range(n)]
    
    # 3. Find the remaining k-1 centers
    for _ in range(1, k):
        next_center_idx = random.choices(range(n), weights=min_sq_distances, k=1)[0]
        new_center = P[next_center_idx]
        C.append(new_center)
        
        # Update min_sq_distances to account for the new center
        for i in range(n):
            dist_to_new = P[i].squared_distance(new_center)
            if dist_to_new < min_sq_distances[i]:
                min_sq_distances[i] = dist_to_new
                
    return C

def kmeansObj(P, C):
    """
    Computes the average squared distance of a point in P to its closest center in C.
    """
    n = len(P)
    if n == 0 or len(C) == 0:
        return 0.0
        
    total_sq_dist = 0.0
    
    for p in P:
        # Find the minimum squared distance from p to any center in C
        min_dist = float('inf')
        for c in C:
            dist = p.squared_distance(c)
            if dist < min_dist:
                min_dist = dist
        total_sq_dist += min_dist
        
    # Return average squared distance
    return total_sq_dist / n


k = 10
k1 = 50

print(f"Running Clustering Tests with k={k}, k1={k1}")

# 1. Run kcenter(P, k) printing its running time
start_t = time.time()
centers_kcenter = kcenter(P, k)
end_t = time.time()
print(f"1--> kcenter(P, {k}) running time: {end_t - start_t:.3f} seconds")

# 2. Run kmeansPP(P, k), then kmeansObj(P, C) printing the returned value
centers_kmeansPP = kmeansPP(P, k)
obj_kmeansPP = kmeansObj(P, centers_kmeansPP)
print(f"2--> kmeansPP(P, {k}) objective value: {obj_kmeansPP:.3f}")

# 3. Run kcenter(P, k1) -> X; kmeansPP(X, k) -> C; kmeansObj(P, C)
X = kcenter(P, k1)
centers_coreset = kmeansPP(X, k)
obj_coreset = kmeansObj(P, centers_coreset)
print(f"3--> Coreset approach (kcenter {k1} -> kmeansPP {k}) objective value: {obj_coreset:.3f}")

Running Clustering Tests with k=10, k1=50
1--> kcenter(P, 10) running time: 0.058 seconds
2--> kmeansPP(P, 10) objective value: 31006.933
3--> Coreset approach (kcenter 50 -> kmeansPP 10) objective value: 419038.401


## Part 2

In [14]:
class MySet:
    def __init__(self):
        self.elements = set()
        
    def addElement(self, element):
        self.elements.add(element)
        
    def union(self, otherSet):
        new_set = MySet()
        new_set.elements = self.elements.union(otherSet.elements)
        return new_set
        
    def intersection(self, otherSet):
        new_set = MySet()
        new_set.elements = self.elements.intersection(otherSet.elements)
        return new_set
        
    def get_elements(self):
        return self.elements


class Position:
    def __init__(self, p, wordIndex):
        """
        p: PageEntry object
        wordIndex: int representing the word's position in the document
        """
        self.p = p
        self.wordIndex = wordIndex
        
    def getPageEntry(self):
        return self.p
        
    def getWordIndex(self):
        return self.wordIndex


class WordEntry:
    def __init__(self, word):
        self.word = word
        self.positions = [] # List of Position objects
        
    def addPosition(self, position):
        self.positions.append(position)
        
    def addPositions(self, positions):
        self.positions.extend(positions)
        
    def getAllPositionsForThisWord(self):
        return self.positions
        
    def getTermFrequency(self, word):
        """
        Returns the raw count (frequency) of this word.
        Since this WordEntry handles one specific word, we just return 
        the length of positions. (Note: TF-IDF calculation will normalize this later if needed).
        """
        return len(self.positions)

In [23]:
import os

class PageIndex:
    def __init__(self):
        # Dictionary mapping a word (string) to its WordEntry object
        self.word_entries = {}
        
    def addPositionForWord(self, word, p):
        # If the word isn't in our index yet, create a new WordEntry for it
        if word not in self.word_entries:
            self.word_entries[word] = WordEntry(word)
        
        # Add the position to the word's entry
        self.word_entries[word].addPosition(p)
        
    def getWordEntries(self):
        return list(self.word_entries.values())


class PageEntry:
    def __init__(self, pageName):
        self.pageName = pageName
        self.pageIndex = PageIndex()
        
        # Exhaustive list of connector words to ignore
        self.connectors = {
            "a", "an", "the", "they", "these", "this", "for", "is", 
            "are", "was", "of", "or", "and", "does", "will", "whose"
        }
        
        # Exhaustive list of plural mappings
        self.plurals = {
            "stacks": "stack",
            "structures": "structure",
            "applications": "application"
        }
        
        # Parse the webpage when the object is instantiated
        self.process_file()
        
    def getPageIndex(self):
        return self.pageIndex
        
    def process_file(self):
        # Assuming the assignment dataset is in a folder called 'webpages'
        filepath = os.path.join("webpages", self.pageName)
        
        try:
            with open(filepath, 'r', encoding='utf-8') as f:
                content = f.read()
                
            # Replace punctuations with spaces
            punctuations = "{}[]<>=().,;\'\"?#!-:"
            for punc in punctuations:
                content = content.replace(punc, ' ')
                
            # Split into words and convert to lowercase
            words = content.lower().split()
            
            # Keep track of word position (1-based indexing)
            word_index = 1 
            
            for word in words:
                # Apply plural to singular transformation if applicable
                if word in self.plurals:
                    word = self.plurals[word]
                    
                # If it's not a connector word, add it to the index
                if word not in self.connectors:
                    pos = Position(self, word_index)
                    self.pageIndex.addPositionForWord(word, pos)
                    
                # Increment index REGARDLESS of whether it was a connector word or not
                word_index += 1
                
        except FileNotFoundError:
            print(f"Error: Could not find webpage '{self.pageName}' in 'webpages' folder.")

In [26]:
class MyHashTable:
    def __init__(self, capacity=1000):
        self.capacity = capacity
        # Array of lists to handle collisions via chaining
        self.table = [[] for _ in range(capacity)]
        
    def getHashIndex(self, word):
        # A simple polynomial rolling hash function for strings
        hash_val = 0
        for char in word:
            hash_val = (hash_val * 31 + ord(char)) % self.capacity
        return hash_val
        
    def addPositionsForWord(self, w: WordEntry):
        idx = self.getHashIndex(w.word)
        bucket = self.table[idx]
        
        # If the word already exists in this bucket, merge the positions
        for existing_entry in bucket:
            if existing_entry.word == w.word:
                existing_entry.addPositions(w.getAllPositionsForThisWord())
                return
                
        # If not found, append the new word entry
        bucket.append(w)
        
    def getWordEntry(self, word):
        # Helper method to retrieve a WordEntry
        idx = self.getHashIndex(word)
        bucket = self.table[idx]
        for entry in bucket:
            if entry.word == word:
                return entry
        return None


class InvertedPageIndex:
    def __init__(self):
        self.hashtable = MyHashTable()
        self.pages_dict = {} # Helper to keep track of valid pages added
        
    def addPage(self, p: PageEntry):
        self.pages_dict[p.pageName] = p
        page_index = p.getPageIndex()
        
        # Iterate through all unique words in this page and add them to the global hash table
        for word_entry in page_index.getWordEntries():
            self.hashtable.addPositionsForWord(word_entry)
            
    def getPagesWhichContainWord(self, word):
        word_entry = self.hashtable.getWordEntry(word)
        page_set = MySet()
        
        if word_entry:
            for pos in word_entry.getAllPositionsForThisWord():
                page_set.addElement(pos.getPageEntry())
                
        return page_set


class SearchEngine:
    def __init__(self):
        self.inverted_index = InvertedPageIndex()
        self.plurals = {"stacks": "stack", "structures": "structure", "applications": "application"}
        
    def performAction(self, actionMessage):
        parts = actionMessage.strip().split()
        if not parts:
            return
            
        action = parts[0]
        
        if action == "addPage":
            page_name = parts[1]
            p = PageEntry(page_name)
            self.inverted_index.addPage(p)
            
        elif action == "queryFindPagesWhichContainWord":
            original_word = parts[1]
            word = original_word.lower()
            
            # Map query word to singular if it's a known plural
            if word in self.plurals:
                word = self.plurals[word]
                
            pages_set = self.inverted_index.getPagesWhichContainWord(word)
            elements = pages_set.get_elements()
            
            if not elements:
                print(f"No webpage contains word {original_word}")
            else:
                # Sort page names alphabetically for clean output
                page_names = sorted([p.pageName for p in elements])
                print(", ".join(page_names))
                
        elif action == "queryFindPositionsOfWordInAPage":
            original_word = parts[1]
            word = original_word.lower()
            page_name = parts[2]
            
            if word in self.plurals:
                word = self.plurals[word]
            
            if page_name not in self.inverted_index.pages_dict:
                print(f"No webpage {page_name} found")
                return
            
            word_entry = self.inverted_index.hashtable.getWordEntry(word)
            
            if not word_entry:
                print(f"Webpage {page_name} does not contain word {original_word}")
                return
                
            positions = []
            for pos in word_entry.getAllPositionsForThisWord():
                if pos.getPageEntry().pageName == page_name:
                    positions.append(pos.getWordIndex())
                    
            if not positions:
                print(f"Webpage {page_name} does not contain word {original_word}")
            else:
                # Sort positions numerically
                print(", ".join(map(str, sorted(positions))))

engine = SearchEngine()

try:
    with open('Q2- webSearch/actions.txt', 'r') as f:
        for line in f:
            if line.strip():
                engine.performAction(line.strip())
except FileNotFoundError:
    print("Error: Could not find 'actions.txt'. Please ensure it's in the same directory.")

No webpage contains word delhi
stack_datastructure_wiki
stack_datastructure_wiki
Webpage stack_datastructure_wiki does not contain word magazines
No webpage contains word allain
stack_cprogramming
stack_cprogramming
stack_cprogramming
stack_oracle
stack_cprogramming, stack_datastructure_wiki, stackoverflow
stackmagazine


In [1]:
import os
import re
from operator import add
from pyspark.sql import SparkSession

# 1. Set the Java path exactly as we found it
os.environ["JAVA_HOME"] = "/opt/homebrew/Cellar/openjdk@11/11.0.30/libexec/openjdk.jdk/Contents/Home" 

In [4]:
import pyspark
local_spark_path = os.path.dirname(pyspark.__file__)
os.environ["SPARK_HOME"] = local_spark_path

In [ ]:
# 2. Initialize Spark Session AFTER setting the path
spark = SparkSession.builder.appName("PageRank").getOrCreate()
sc = spark.sparkContext

current_dir = os.getcwd()

def run_pagerank(filename, beta=0.8, iterations=40):
    # Construct the full local file path
    # We use 'file://' + absolute path to bypass HDFS settings
    file_path = "file://" + os.path.join(current_dir, filename)
    print(f"--- Running PageRank on {file_path} ---")
    
    # Load the dataset using the local path
    lines = sc.textFile(file_path)
    
    edges = lines.map(lambda line: tuple(re.split(r'\s+', line.strip()))) \
                 .filter(lambda x: len(x) == 2) \
                 .distinct()
                 
    # Find all unique nodes to calculate 'n'
    nodes = edges.map(lambda x: x[0]).union(edges.map(lambda x: x[1])).distinct()
    n = nodes.count()
    print(f"Total nodes (n) = {n}")
    
    # Create the adjacency list: (source, [dest1, dest2, ...])
    links = edges.groupByKey().mapValues(list).cache()
    
    # Initialize ranks to 1/n
    ranks = nodes.map(lambda node: (node, 1.0 / n))
    
    # Iterative PageRank Calculation
    for iteration in range(iterations):
        
        def compute_contribs(node_urls_rank):
            _, (urls, rank) = node_urls_rank
            num_urls = len(urls)
            for url in urls:
                yield (url, rank / num_urls)
                
        contribs = links.join(ranks).flatMap(compute_contribs)
        summed_contribs = contribs.reduceByKey(add)
        
        ranks = nodes.map(lambda x: (x, 0.0)) \
                     .leftOuterJoin(summed_contribs) \
                     .mapValues(lambda x: (1.0 - beta) / n + beta * (x[1] if x[1] is not None else 0.0))
                     
    # Collect and sort the final ranks
    sorted_ranks = ranks.sortBy(lambda x: x[1], ascending=False).collect()
    
    print("\nTop 5 nodes with the highest scores:")
    for i in range(5):
        if i < len(sorted_ranks):
            print(f"Node {sorted_ranks[i][0]}: {sorted_ranks[i][1]:.6f}")
            
    print("\nBottom 5 nodes with the lowest scores:")
    for i in range(1, 6):
        if i <= len(sorted_ranks):
            print(f"Node {sorted_ranks[-i][0]}: {sorted_ranks[-i][1]:.6f}")
            
    return sorted_ranks

print("Testing on small.txt...")
try:
    small_ranks = run_pagerank("small.txt", beta=0.8, iterations=40)
except Exception as e:
    print(f"Error running on small.txt: {e}")

print("\n" + "="*50 + "\n")

Testing on small.txt...
--- Running PageRank on file:///Users/admin/Desktop/Assign_4_MLBD/small.txt ---


Total nodes (n) = 100



Top 5 nodes with the highest scores:
Node 53: 0.035731
Node 14: 0.034171
Node 40: 0.033630
Node 1: 0.030006
Node 27: 0.029720

Bottom 5 nodes with the lowest scores:
Node 85: 0.003410
Node 59: 0.003670
Node 81: 0.003695
Node 37: 0.003808
Node 89: 0.003922




In [9]:
print("Running on whole.txt...")
try:
     whole_ranks = run_pagerank("whole.txt", beta=0.8, iterations=40)
except Exception as e:
     print(f"Error running on whole.txt: {e}")

Running on whole.txt...
--- Running PageRank on file:///Users/admin/Desktop/Assign_4_MLBD/whole.txt ---
Total nodes (n) = 1000



Top 5 nodes with the highest scores:
Node 263: 0.002020
Node 537: 0.001943
Node 965: 0.001925
Node 243: 0.001853
Node 285: 0.001827

Bottom 5 nodes with the lowest scores:
Node 558: 0.000329
Node 93: 0.000351
Node 62: 0.000353
Node 424: 0.000355
Node 408: 0.000388
